# **Import Modules**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# **Data Understanding**

In [3]:
train = pd.read_csv('train.csv')

test = pd.read_csv('test.csv')

bank_full = pd.read_csv('bank-full.csv', sep = ';')

In [4]:
train.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [5]:
bank_full.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [6]:
train['job'].unique()

array(['technician', 'blue-collar', 'student', 'admin.', 'management',
       'entrepreneur', 'self-employed', 'unknown', 'services', 'retired',
       'housemaid', 'unemployed'], dtype=object)

In [7]:
train['education'].unique()

array(['secondary', 'primary', 'tertiary', 'unknown'], dtype=object)

In [8]:
train['marital'].unique()

array(['married', 'single', 'divorced'], dtype=object)

In [9]:
train['default'].unique()

array(['no', 'yes'], dtype=object)

In [10]:
train['housing'].unique()

array(['no', 'yes'], dtype=object)

In [11]:
train['loan'].unique()

array(['no', 'yes'], dtype=object)

In [12]:
train['contact'].unique()

array(['cellular', 'unknown', 'telephone'], dtype=object)

In [13]:
train['month'].unique()

array(['aug', 'jun', 'may', 'feb', 'apr', 'nov', 'jul', 'jan', 'oct',
       'mar', 'sep', 'dec'], dtype=object)

In [14]:
train['poutcome'].unique()


array(['unknown', 'other', 'failure', 'success'], dtype=object)

In [41]:
train['y'].unique()

array([0, 1], dtype=int64)

In [16]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 18 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         750000 non-null  int64 
 1   age        750000 non-null  int64 
 2   job        750000 non-null  object
 3   marital    750000 non-null  object
 4   education  750000 non-null  object
 5   default    750000 non-null  object
 6   balance    750000 non-null  int64 
 7   housing    750000 non-null  object
 8   loan       750000 non-null  object
 9   contact    750000 non-null  object
 10  day        750000 non-null  int64 
 11  month      750000 non-null  object
 12  duration   750000 non-null  int64 
 13  campaign   750000 non-null  int64 
 14  pdays      750000 non-null  int64 
 15  previous   750000 non-null  int64 
 16  poutcome   750000 non-null  object
 17  y          750000 non-null  int64 
dtypes: int64(9), object(9)
memory usage: 103.0+ MB


In [17]:
#train_full = pd.concat([train, bank_full], axis=0, ignore_index=True).drop_duplicates()

In [18]:
#train_full.head()

# **Data Pre-Processing**

In [19]:
#train_full.shape

In [20]:
test.shape

(250000, 17)

In [21]:
#rain_full['y'].unique()

In [22]:
#X = train_full.drop(['id','y'], axis=1)
#y = train_full['y'].replace({'yes':1,'no':0}).astype(int)

X = train.drop(['id','y'], axis=1)
y = train['y']#.replace({'yes':1,'no':0}).astype(int)


In [23]:
X_test_final = test.drop(columns=['id'])

In [24]:
binary_yn = ["default", "housing", "loan"]

ordinal_cols = ["education", "month"]

nominal_cat = ["job", "marital", "contact", "poutcome"]

numeric_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]


In [25]:
# Define category order for ordinals
edu_order = ["unknown", "primary", "secondary", "tertiary"]
month_order = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]

In [26]:
def map_yes_no(df):
    mapping = {"yes": 1, "no": 0}
    out = df.copy()
    for c in binary_yn:
        out[c] = out[c].map(mapping).astype(int)
    return out


In [27]:
yn_mapper = FunctionTransformer(map_yes_no, feature_names_out="one-to-one")

In [28]:
binary_pipe = Pipeline([("yn_map", yn_mapper)])

nominal_pipe = Pipeline([("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

numeric_pipe_scaled = Pipeline([("scaler", StandardScaler())])

ordinal_pipe = Pipeline([
    ("ord_enc", OrdinalEncoder(
        categories=[edu_order, month_order],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        ("bin", binary_pipe, binary_yn),
        ("ordinal", ordinal_pipe, ordinal_cols),   # now using pipeline
        ("nom", nominal_pipe, nominal_cat),
        ("num", numeric_pipe_scaled, numeric_cols)
    ],
    remainder="drop"
)

In [30]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# **Helper Functions**

In [31]:
def run_search_rand(pipe, param_dist, name):
    rand = RandomizedSearchCV(pipe, param_dist, n_iter=20, cv=5, n_jobs=-1, random_state=42, scoring="accuracy")
    rand.fit(X_train, y_train)
    print(f"Best model for {name}:")
    print("Best Params:", rand.best_params_)
    return rand

def run_search_grid(pipe, params, name):
    grid = GridSearchCV(pipe, params, cv=5, n_jobs=-1, scoring="accuracy")
    grid.fit(X_train, y_train)
    print(f"\n✅ Finished GridSearch for {name}")
    print("Best Params:", grid.best_params_)
    return grid

# **Evaluation Function**

In [32]:
def evaluate_model(search, model_name, X, y, X_valid, y_valid):
    best_model = search.best_estimator_
    val_score = cross_val_score(best_model, X, y, cv=5).mean()
    test_score = accuracy_score(y_valid, best_model.predict(X_valid))
    gap = val_score - test_score
    if gap > 0.05 and val_score >= 0.85:
        fit_msg = "🚨 Overfitting"
    elif val_score < 0.70 and test_score < 0.70:
        fit_msg = "⚠️ Underfitting"
    elif abs(gap) <= 0.05 and test_score >= 0.75:
        fit_msg = "✅ Good Fit"
    else:
        fit_msg = "ℹ️ Borderline"
    print(f"\n--- {model_name} ---")
    print("Validation Accuracy:", round(val_score,3))
    print("Test Accuracy:", round(test_score,3))
    print("Fit:", fit_msg)
    return {"Model": model_name, "Validation_Accuracy": round(val_score,3), "Test_Accuracy": round(test_score,3), "Fit": fit_msg}

In [33]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

def evaluate_model(model, model_name, X, y, X_valid, y_valid):
    # Cross-validation score on training set
    val_score = cross_val_score(model, X, y, cv=5).mean()
    
    # Test score on validation set
    test_score = accuracy_score(y_valid, model.predict(X_valid))
    
    # Gap between train-val
    gap = val_score - test_score
    
    # Fit diagnosis
    if gap > 0.05 and val_score >= 0.85:
        fit_msg = "🚨 Overfitting"
    elif val_score < 0.70 and test_score < 0.70:
        fit_msg = "⚠️ Underfitting"
    elif abs(gap) <= 0.05 and test_score >= 0.75:
        fit_msg = "✅ Good Fit"
    else:
        fit_msg = "ℹ️ Borderline"
    
    print(f"\n--- {model_name} ---")
    print("Validation Accuracy:", round(val_score,3))
    print("Test Accuracy:", round(test_score,3))
    print("Fit:", fit_msg)
    
    return {
        "Model": model_name,
        "Validation_Accuracy": round(val_score,3),
        "Test_Accuracy": round(test_score,3),
        "Fit": fit_msg
    }


# **HyperParameter Tuning For XGBoost**

In [36]:
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators = 5000,
        max_depth = 6,
        random_state=42,
        eval_metric="logloss",
        tree_method="hist"
    ))
])


In [39]:
xgb = xgb_pipe.fit(X_train, y_train)

In [40]:
results = evaluate_model(xgb, "XGBoost", X_train, y_train, X_valid, y_valid)


--- XGBoost ---
Validation Accuracy: 0.93
Test Accuracy: 0.931
Fit: ✅ Good Fit


In [43]:
results

{'Model': 'XGBoost',
 'Validation_Accuracy': 0.93,
 'Test_Accuracy': 0.931,
 'Fit': '✅ Good Fit'}

In [44]:
# Predict using the fitted pipeline
final_preds = xgb.predict(X_test_final)

In [45]:
# Build submission DataFrame
submission = pd.DataFrame({
    "id": test["id"],
    "y": final_preds   # make sure 'y' matches competition requirements
})

In [46]:
# Save to CSV
submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")

submission.csv saved!


In [47]:
import pickle

# Save the trained pipeline
with open("bank_classifier.pkl", "wb") as f:
    pickle.dump(xgb, f)

print("Pipeline saved as bank_classifier.pkl")


Pipeline saved as bank_classifier.pkl
